# DLC Spatial Analysis (Reward-Aligned)

Spatial analysis of DLC tracking data aligned to reward events. Extends Phase 5's whole-session analyses with peri-reward windowing.

Outputs go to `stroke_orofacial_pipeline_outputs/dlc_analysis/spatial_reward_aligned/` on MICROSCOPE.

**Phase 6.7 update:** orchestration now flows through `SessionManifest`. Cell 3 builds one manifest per (animal, date); cell 4 iterates and passes each manifest to `build_reward_aligned_analysis()`. Phase 6 is DLC-required — manifests with no DLC files raise FileNotFoundError, surfaced in the per-(bodypart, mode, group, window) error log.

**Per Priya 2026-04-20:** the rendered PNG is unchanged, but `HEATMAP_NBINS` is now exposed as a USER INPUT for callers who want to override the binning resolution at the analysis point.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46"]

# Dates to process. None = all dates in session range.
DATES = None

# Bodyparts and centering modes
BODYPARTS = ["tongue", "jaw", "nose"]
CENTER_MODES = ["nose", "spout_midpoint", "eye_midpoint"]

# Reward windows (ms): [pre_ms, post_ms]
REWARD_WINDOWS_MS = [(-500, 5000), (-100, 1000)]

# Trial groups
TRIAL_GROUPS = ["all", "L", "R"]

# Heatmap binning resolution (display only — reward-aligned analyses
# already use the dlc_spatial_reward_aligned.default_nbins from config
# for the histogram itself; this is reserved for future per-call overrides).
HEATMAP_NBINS = 100

# Overwrite mode: "skip" or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.dlc_spatial.tone_aligned import (
    build_reward_aligned_analysis,
    save_reward_aligned_npz,
)
from stroke_orofacial.manifest import build_manifest
from stroke_orofacial.paths import PathResolver

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
spatial_config = defaults["dlc_spatial"]
reward_config = defaults["dlc_spatial_reward_aligned"]
output_root = resolver.pipeline_outputs("dlc_analysis") / "spatial_reward_aligned"

In [ ]:
# ============================================================
# Build a SessionManifest per (animal, date). Reward-aligned
# analysis needs DLC + alignment_template + Phase 2 licks NPZ;
# the manifest surfaces the first two so missing inputs are visible.
# ============================================================

manifests = []  # list of (animal_id, date, manifest)

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for date in dates:
        manifest = build_manifest(animal_id, date, resolver, info)
        manifests.append((animal_id, date, manifest))

n_with_dlc = sum(1 for _, _, m in manifests if m.dlc_h5_files)
n_with_align = sum(1 for _, _, m in manifests if m.alignment_template is not None)
print(f"Built {len(manifests)} manifests:")
print(f"  with DLC files:           {n_with_dlc}")
print(f"  with alignment template:  {n_with_align}")

In [ ]:
# ============================================================
# For each (manifest, center_mode, bodypart, window, trial_group):
# call build_reward_aligned_analysis(manifest, ...) and render the PNG.
# ============================================================

for center_mode in CENTER_MODES:
    for bodypart in BODYPARTS:
        for window_ms in REWARD_WINDOWS_MS:
            for trial_group in TRIAL_GROUPS:
                n_ok, n_skip, n_err = 0, 0, 0
                n_skip_no_rewards, n_skip_no_frames = 0, 0
                win_tag = f"{int(window_ms[0])}to{int(window_ms[1])}ms"

                for animal_id, date, manifest in manifests:
                    date_str = date.strftime("%Y%m%d")
                    out_dir = (
                        output_root / f"{bodypart}_{center_mode}"
                        / f"trials-{trial_group}_{win_tag}" / animal_id
                    )
                    out_path = out_dir / f"heatmap_{date_str}.png"

                    if OVERWRITE == "skip" and out_path.exists():
                        n_skip += 1
                        continue

                    try:
                        # Phase 6.8a: pull the per-animal L/R flag from
                        # AnimalInfo and forward it so (a) the Phase 2
                        # stamp is cross-checked and (b) the Phase 6
                        # NPZ gets stamped with the same convention.
                        animal_info = animals_config[animal_id]
                        result = build_reward_aligned_analysis(
                            manifest, bodypart, center_mode, trial_group,
                            window_ms, resolver,
                            spatial_config, reward_config,
                            wavesurfer_sides_match_mouse_sides=(
                                animal_info.wavesurfer_sides_match_mouse_sides
                            ),
                        )
                        # Phase 6.8a.3: split the two legitimate-empty
                        # skip conditions so the emit log distinguishes
                        # them. `result is None` = zero rewards for this
                        # trial_group (valid session, just no L/R/all);
                        # `result["histogram"] is None` = DLC likelihood
                        # falls below threshold on every frame in the
                        # requested window.
                        if result is None:
                            n_skip_no_rewards += 1
                            logging.info(
                                "skip_no_rewards %s/%s/%s/%s/%s/%s",
                                animal_id, date_str, bodypart, center_mode,
                                trial_group, win_tag,
                            )
                            continue
                        if result["histogram"] is None:
                            n_skip_no_frames += 1
                            logging.info(
                                "skip_no_frames %s/%s/%s/%s/%s/%s",
                                animal_id, date_str, bodypart, center_mode,
                                trial_group, win_tag,
                            )
                            continue

                        hist = result["histogram"]
                        fig, ax = plt.subplots(figsize=(4.6, 4.4))
                        if hist["H_counts"].sum() > 0:
                            ax.imshow(
                                hist["H_proportion"].T,
                                extent=hist["extent"],
                                origin="lower", aspect="auto",
                                cmap="magma",
                                norm=LogNorm(vmin=1e-5),
                            )
                        ax.set_title(
                            f"{animal_id} {bodypart} {date_str}\n"
                            f"{center_mode} | {trial_group} | {win_tag}\n"
                            f"rewards={result['n_rewards']}, frames={result['n_selected_frames']}"
                        )
                        ax.set_xlabel(f"{bodypart}_x")
                        ax.set_ylabel(f"{bodypart}_y")
                        out_path.parent.mkdir(parents=True, exist_ok=True)
                        fig.savefig(str(out_path), dpi=200, bbox_inches="tight")
                        plt.close(fig)

                        # Phase 6.8a: also save the histogram as NPZ so
                        # downstream stats / drift diagnostics can read
                        # H_counts programmatically. The helper stamps
                        # the per-animal L/R flag from `result` so the
                        # trial_group label is unambiguous at read time.
                        save_reward_aligned_npz(
                            result, out_path.with_suffix(".npz"),
                        )

                        n_ok += 1
                    except FileNotFoundError as e:
                        n_err += 1
                        logging.warning(
                            "%s/%s/%s/%s/%s: %s",
                            animal_id, date_str, bodypart, center_mode, trial_group, e,
                        )
                    except Exception as e:
                        n_err += 1
                        logging.warning(
                            "%s/%s/%s/%s/%s: %s",
                            animal_id, date_str, bodypart, center_mode, trial_group, e,
                        )

                tag = f"{bodypart}/{center_mode}/trials-{trial_group}/{win_tag}"
                if n_ok + n_skip + n_err + n_skip_no_rewards + n_skip_no_frames > 0:
                    print(
                        f"  {tag}: ok={n_ok} skip={n_skip} "
                        f"skip_no_rewards={n_skip_no_rewards} "
                        f"skip_no_frames={n_skip_no_frames} err={n_err}"
                    )